# Arabic Handwriting — Resume Training

**Before running:** make sure all 3 input datasets are attached:
- `arabic-handwriting-data`
- `arabic-handwriting-checkpoints`
- `arabic-handwriting-code`

Accelerator must be set to **GPU T4 x2** or **P100**.

In [ ]:
import os, shutil
from pathlib import Path

BASE       = Path('/kaggle/input/datasets/mennadafrawy')
ML         = Path('/kaggle/working/ml')
CODE_INPUT = BASE / 'arabic-handwriting-code'
CKPT_INPUT = BASE / 'arabic-handwriting-checkpoints'
DATA_INPUT = BASE / 'arabic-handwriting-data'

# Clean previous run
shutil.rmtree(ML, ignore_errors=True)

for d in [
    ML / 'data',
    ML / 'outputs/arabic_v1/checkpoints',
    ML / 'outputs/arabic_v1/config',
    ML / 'outputs/arabic_v1/logs',
    ML / 'outputs/arabic_v1/plots/training',
    ML / 'outputs/arabic_v1/plots/samples',
]:
    d.mkdir(parents=True, exist_ok=True)

shutil.copytree(CODE_INPUT / 'src', ML / 'src')
shutil.copytree(CODE_INPUT / 'config', ML / 'config')

data_src = DATA_INPUT / 'processed' if (DATA_INPUT / 'processed').exists() else DATA_INPUT
os.symlink(str(data_src), str(ML / 'data/processed'))

ckpt_dst = ML / 'outputs/arabic_v1/checkpoints'
for f in CKPT_INPUT.glob('model-*'):
    shutil.copy2(f, ckpt_dst / f.name)
    print(f'Copied checkpoint: {f.name}')

cfg_src = CKPT_INPUT / 'config.yaml'
shutil.copy2(cfg_src if cfg_src.exists() else ML / 'config/config.yaml',
             ML / 'outputs/arabic_v1/config/config.yaml')

print('Setup complete.')

In [ ]:
!pip install pydantic pyyaml python-dotenv wandb -q

In [ ]:
# Patience [1000, 700, 500] — longer convergence per phase for better character quality
import yaml
from pathlib import Path

for cfg_path in [
    Path('/kaggle/working/ml/config/config.yaml'),
    Path('/kaggle/working/ml/outputs/arabic_v1/config/config.yaml'),
]:
    with open(cfg_path) as f:
        cfg = yaml.safe_load(f)
    cfg['training_params']['patiences'] = [1000, 700, 500]
    with open(cfg_path, 'w') as f:
        yaml.dump(cfg, f)

print(f"Patiences set to: {cfg['training_params']['patiences']}")

In [ ]:
import json, os, shutil, subprocess, threading, time, torch
from pathlib import Path

# --- Kaggle credentials from Secrets (Add-ons -> Secrets) ---
try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    cred = {"username": s.get_secret("KAGGLE_USERNAME"), "key": s.get_secret("KAGGLE_KEY")}
    kdir = Path.home() / '.kaggle'
    kdir.mkdir(exist_ok=True)
    (kdir / 'kaggle.json').write_text(json.dumps(cred))
    os.chmod(kdir / 'kaggle.json', 0o600)
    print("Kaggle credentials ready — auto-save enabled")
except Exception as e:
    print(f"WARNING: Kaggle auth failed ({e}) — auto-save disabled")

CKPT_DIR = Path('/kaggle/working/ml/outputs/arabic_v1/checkpoints')
STAGING  = Path('/kaggle/working/ckpt_staging')

def get_best_and_latest(ckpt_dir):
    """Return (best_checkpoint_path, latest_checkpoint_path) by val_loss and step."""
    checkpoints = list(ckpt_dir.glob('model-*'))
    if not checkpoints:
        return None, None
    latest = max(checkpoints, key=lambda f: int(f.name.split('-')[1]))
    best = latest  # fallback: latest
    best_loss = float('inf')
    for ckpt in checkpoints:
        try:
            data = torch.load(ckpt, map_location='cpu', weights_only=False)
            loss = data.get('val_loss', float('inf'))
            if loss < best_loss:
                best_loss = loss
                best = ckpt
        except Exception:
            pass
    return best, latest

def push_checkpoints():
    best, latest = get_best_and_latest(CKPT_DIR)
    if best is None:
        print("[auto-save] no checkpoints yet, skipping")
        return
    STAGING.mkdir(exist_ok=True)
    for f in STAGING.glob('model-*'):
        f.unlink()
    # always push latest; also push best if different
    files_to_push = {latest}
    if best != latest:
        files_to_push.add(best)
    for ckpt in files_to_push:
        shutil.copy2(ckpt, STAGING / ckpt.name)
    shutil.copy2(Path('/kaggle/working/ml/outputs/arabic_v1/config/config.yaml'), STAGING / 'config.yaml')
    names = ', '.join(f.name for f in files_to_push)
    (STAGING / 'dataset-metadata.json').write_text(json.dumps({
        "title": "Arabic Handwriting Checkpoints",
        "id": "mennadafrawy/arabic-handwriting-checkpoints",
        "licenses": [{"name": "CC0-1.0"}]
    }))
    r = subprocess.run(
        ['kaggle', 'datasets', 'version', '-p', str(STAGING), '-m', f'auto-save: {names}'],
        capture_output=True, text=True
    )
    print(f"[auto-save] pushed {names}: {(r.stdout or r.stderr).strip()}")

def watcher(interval=3600):
    while True:
        time.sleep(interval)
        try:
            push_checkpoints()
        except Exception as e:
            print(f"[auto-save] error: {e}")

threading.Thread(target=watcher, daemon=True).start()
print("Auto-save watcher started — pushes best + latest checkpoint every hour")

In [ ]:
import os
os.environ['RANK'] = '0'
os.environ['WORLD_SIZE'] = '1'
os.environ['LOCAL_RANK'] = '0'
os.chdir('/kaggle/working/ml')

!python -m src.train --run_name arabic_v1

In [ ]:
# Run after training to see saved checkpoints and their val losses
import torch
from pathlib import Path

ckpt_dir = Path('/kaggle/working/ml/outputs/arabic_v1/checkpoints')
rows = []
for f in sorted(ckpt_dir.glob('model-*'), key=lambda x: int(x.name.split('-')[1])):
    try:
        d = torch.load(f, map_location='cpu', weights_only=False)
        rows.append((f.name, d.get('val_loss', '?'), d.get('restart_idx', '?')))
    except Exception as e:
        rows.append((f.name, f'err: {e}', '?'))
for name, loss, phase in rows:
    print(f'{name:20s}  val_loss={loss:<12}  phase={phase}')